In [ ]:
!pip install torch torchaudio librosa soundfile pesq matplotlib --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.2 MB/s eta 0:00:00


In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import numpy as np
import librosa
import soundfile as sf
import random
import json

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from IPython.display import Audio, display
from tqdm import tqdm

BASE_PATH = "/content/drive/MyDrive/colab-data/audio-denoising"

# Original input directories (the ones containing the full-length .flac files)
CLEAN_AUDIO_PATH = os.path.join(BASE_PATH, "LibriSpeech", "test-clean")
NOISY_AUDIO_PATH = os.path.join(BASE_PATH, "data", "noisy")

# Output directories for 2s segments
OUT_CLEAN_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "clean")
OUT_NOISY_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "noisy")


In [ ]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

os.makedirs(OUT_CLEAN_DIR, exist_ok=True)
os.makedirs(OUT_NOISY_DIR, exist_ok=True)

def gather_flac_map(folder):
    """
    Recursively collect all .flac paths under 'folder',
    store them in a dict keyed by the file’s basename.
    """
    flac_map = {}
    for root, dirs, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith(".flac"):
                full_path = os.path.join(root, fname)
                flac_map[fname] = full_path
    return flac_map

def create_2sec_segments(clean_dir, noisy_dir, out_clean_dir, out_noisy_dir, sr=16000):
    """
    1) Match clean & noisy .flac by basename
    2) For each pair, split both waveforms into multiple 2-second segments
       (i.e. 2*sr = 32000 samples).
    3) Skip leftover <2s at the end of each file.
    4) Write each pair of segments with a suffix "_seg0.flac", etc.
    """
    seg_len = 2 * sr  # 2 seconds of audio at 16k

    # 1) Gather maps
    clean_map = gather_flac_map(clean_dir)
    noisy_map = gather_flac_map(noisy_dir)

    # 2) Intersection by basename
    common_names = sorted(set(clean_map.keys()).intersection(set(noisy_map.keys())))
    print(f"Found {len(common_names)} matched .flac basenames.")

    # 3) For each common file, produce segments
    for basename in tqdm(common_names, desc="Splitting into 2s segments"):
        clean_path = clean_map[basename]
        noisy_path = noisy_map[basename]

        clean_audio, _ = librosa.load(clean_path, sr=sr)
        noisy_audio, _ = librosa.load(noisy_path, sr=sr)

        # We'll only produce segments up to the min length
        L = min(len(clean_audio), len(noisy_audio))
        num_segments = L // seg_len  # skip leftover

        for i in range(num_segments):
            start = i * seg_len
            end = start + seg_len

            seg_clean = clean_audio[start:end]
            seg_noisy = noisy_audio[start:end]

            # Example filename: "myfile.flac" -> "myfile_seg0.flac"
            out_clean_name = f"{basename.replace('.flac','')}_seg{i}.flac"
            out_noisy_name = f"{basename.replace('.flac','')}_seg{i}.flac"

            sf.write(os.path.join(out_clean_dir, out_clean_name), seg_clean, sr, format="FLAC")
            sf.write(os.path.join(out_noisy_dir, out_noisy_name), seg_noisy, sr,  format="FLAC")

# Actually create the 2s segments
create_2sec_segments(
    CLEAN_AUDIO_PATH,
    NOISY_AUDIO_PATH,
    OUT_CLEAN_DIR,
    OUT_NOISY_DIR,
    sr=16000
)

print("Finished creating 2-second files!")


Found 2620 matched .flac basenames.


Splitting into 2s segments: 100%|██████████| 2620/2620 [04:52<00:00,  8.97it/s]

Finished creating 2-second files!


In [ ]:
# sample two sec audio
import os

OUT_CLEAN_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "clean")
OUT_NOISY_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "noisy")

#get first flac file from both path
clean_file = os.listdir(OUT_CLEAN_DIR)[0]
noisy_file = os.listdir(OUT_NOISY_DIR)[0]

#load audio
clean_audio, _ = librosa.load(os.path.join(OUT_CLEAN_DIR, clean_file), sr=16000)
noisy_audio, _ = librosa.load(os.path.join(OUT_NOISY_DIR, noisy_file), sr=16000)

print(f"Clean audio shape: {clean_audio.shape}")
display(Audio(clean_audio, rate=16000))
print(f"Noisy audio shape: {noisy_audio.shape}")
display(Audio(noisy_audio, rate=16000))

Clean audio shape: (32000,)


Noisy audio shape: (32000,)


In [ ]:
import os
import numpy as np
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from IPython.display import Audio, display
from tqdm import tqdm

##########################
# CONFIG FOR 2-SECOND RUN
##########################
class Config:
    sr = 16000
    # Now each file is exactly 2 seconds, so 32000 samples:
    segment_length = 32000
    n_fft = 512
    hop_length = 128
    win_length = 512

    batch_size = 8
    lr = 1e-4
    epochs = 10
    alpha = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

config = Config()

# We read from the newly created 2-second segments
BASE_PATH = "/content/drive/MyDrive/colab-data/audio-denoising"
CLEAN_2SEC_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "clean")
NOISY_2SEC_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "noisy")


##########################
# DATASET
##########################
class TwoSecDataset(Dataset):
    """
    - Gathers matched 2-second .flac files from "clean" and "noisy" directories
      by comparing leaf filenames.
    - Each file is already exactly 2s, so we just load them directly
      (no random slicing needed).
    - Convert to STFT log-magnitude, scale to [0..1], pass to model.
    """
    def __init__(self, clean_dir, noisy_dir, conf):
        super().__init__()
        self.conf = conf

        # Build map: basename -> full path
        self.clean_map = self._gather_flac_map(clean_dir)
        self.noisy_map = self._gather_flac_map(noisy_dir)

        # Intersection by leaf name
        common = sorted(set(self.clean_map.keys()).intersection(self.noisy_map.keys()))
        if not common:
            raise ValueError("No matching 2-second .flac basenames found!")
        self.clean_files = [self.clean_map[name] for name in common]
        self.noisy_files = [self.noisy_map[name] for name in common]

        # Basic sanity check
        for c,n in zip(self.clean_files, self.noisy_files):
            if os.path.basename(c) != os.path.basename(n):
                raise ValueError(f"Filename mismatch: {c} vs {n}")

        print(f"Found {len(self.clean_files)} matched 2s pairs.")
        # Precompute global min/max from a random subset
        print("Computing global min/max across STFT magnitudes ...")
        self.global_min, self.global_max = self._compute_minmax(num_samples=200)
        print(f"  => global min={self.global_min:.4f}, max={self.global_max:.4f}")

    def __len__(self):
        return len(self.clean_files)

    def _gather_flac_map(self, folder):
        flac_map = {}
        for root, dirs, files in os.walk(folder):
            for fname in files:
                if fname.lower().endswith(".flac"):
                    flac_map[fname] = os.path.join(root, fname)
        return flac_map

    def _compute_minmax(self, num_samples):
        import random
        indices = random.sample(range(len(self.clean_files)),
                                min(num_samples, len(self.clean_files)))
        mags = []
        for i in tqdm(indices):
            # load, stft -> logmag
            c_wav, _ = librosa.load(self.clean_files[i], sr=self.conf.sr)
            n_wav, _ = librosa.load(self.noisy_files[i], sr=self.conf.sr)

            c_logmag = self._stft_logmag(c_wav)
            n_logmag = self._stft_logmag(n_wav)

            mags.append(c_logmag)
            mags.append(n_logmag)

        cat = np.concatenate([x.flatten() for x in mags])
        return float(cat.min()), float(cat.max())

    def _stft_logmag(self, audio):
        D = librosa.stft(
            audio,
            n_fft=self.conf.n_fft,
            hop_length=self.conf.hop_length,
            win_length=self.conf.win_length
        )
        mag = np.abs(D)
        return np.log1p(mag)

    def __getitem__(self, idx):
        c_path = self.clean_files[idx]
        n_path = self.noisy_files[idx]

        c_wav, _ = librosa.load(c_path, sr=self.conf.sr)
        n_wav, _ = librosa.load(n_path, sr=self.conf.sr)

        # Both should be exactly 2s => 32000 samples
        c_logmag = self._stft_logmag(c_wav)
        n_logmag = self._stft_logmag(n_wav)

        # Scale
        mn, mx = self.global_min, self.global_max
        c_norm = (c_logmag - mn) / (1e-8 + (mx - mn))
        n_norm = (n_logmag - mn) / (1e-8 + (mx - mn))

        # shape => [1, freq, time]
        c_norm = c_norm[np.newaxis, ...]
        n_norm = n_norm[np.newaxis, ...]

        return {
            "clean_audio": torch.FloatTensor(c_wav),   # [32000]
            "noisy_audio": torch.FloatTensor(n_wav),   # [32000]
            "clean_mag":   torch.FloatTensor(c_norm),  # [1, freq, time]
            "noisy_mag":   torch.FloatTensor(n_norm),  # [1, freq, time]
        }



class SimpleUNet(nn.Module):
    """
    Same "time-downsampling" U-Net
    """
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2)
        )
        self.pool1 = nn.MaxPool2d(kernel_size=(1,2))

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1), nn.LeakyReLU(0.2)
        )
        self.pool2 = nn.MaxPool2d(kernel_size=(1,2))

        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, padding=1), nn.LeakyReLU(0.2),
        )

        self.up2 = nn.Conv2d(128, 64, 1)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64+64, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1), nn.LeakyReLU(0.2),
        )

        self.up1 = nn.Conv2d(64, 32, 1)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32+32, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        u2_up = F.interpolate(u2, size=(e2.shape[2], e2.shape[3]), mode='nearest')
        d2 = self.dec2(torch.cat([u2_up, e2], dim=1))

        u1 = self.up1(d2)
        u1_up = F.interpolate(u1, size=(e1.shape[2], e1.shape[3]), mode='nearest')
        d1 = self.dec1(torch.cat([u1_up, e1], dim=1))
        return d1


##########################
# TRAINING
##########################
def train_one_epoch(model, loader, optimizer, conf, dataset, epoch_idx=1):
    model.train()
    total_loss = 0.0
    alpha = conf.alpha
    gmin, gmax = dataset.global_min, dataset.global_max
    seglen = conf.segment_length

    loop = tqdm(loader, desc=f"Epoch {epoch_idx} [training]", leave=False)
    for batch in loop:
        noisy_mag = batch["noisy_mag"].to(DEVICE)
        clean_mag = batch["clean_mag"].to(DEVICE)
        noisy_audio = batch["noisy_audio"].to(DEVICE)
        clean_audio = batch["clean_audio"].to(DEVICE)

        optimizer.zero_grad()

        # Forward
        pred_mag = model(noisy_mag)  # [B,1,freq,time]
        loss_mag = F.l1_loss(pred_mag, clean_mag)

        # time-domain reconstruction
        pred_logmag_unscaled = pred_mag*(gmax - gmin) + gmin
        lin_pred_mag = torch.expm1(pred_logmag_unscaled)  # revert log1p

        stft_noisy = torch.stft(
            noisy_audio,
            n_fft=conf.n_fft,
            hop_length=conf.hop_length,
            win_length=conf.win_length,
            return_complex=True
        )
        phase_noisy = torch.angle(stft_noisy)
        lin_pred_mag = lin_pred_mag.squeeze(1)  # [B,freq,time]
        stft_pred = lin_pred_mag * torch.exp(1j*phase_noisy)

        pred_audio = torch.istft(
            stft_pred,
            n_fft=conf.n_fft,
            hop_length=conf.hop_length,
            win_length=conf.win_length,
            length=seglen
        )

        loss_time = F.l1_loss(pred_audio, clean_audio)

        loss = alpha*loss_mag + (1-alpha)*loss_time
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss/len(loader)


def validate_on_sample(model, dataset, idx=0):
    model.eval()
    with torch.no_grad():
        item = dataset[idx]
        c_wav = item["clean_audio"].numpy()
        n_wav = item["noisy_audio"].numpy()

        c_mag = item["clean_mag"].numpy()
        n_mag = item["noisy_mag"].numpy()

        inp = torch.FloatTensor(n_mag).unsqueeze(0).to(DEVICE)  # [1,1,freq,time]
        pred = model(inp).squeeze(0).cpu().numpy()              # [1,freq,time]

        # unscale
        gmin, gmax = dataset.global_min, dataset.global_max
        pred_logmag = pred[0]*(gmax - gmin) + gmin
        lin_pred_mag = np.expm1(pred_logmag)

        # combine with noisy phase
        stft_noisy = librosa.stft(n_wav,
                                  n_fft=dataset.conf.n_fft,
                                  hop_length=dataset.conf.hop_length,
                                  win_length=dataset.conf.win_length)
        phase_noisy = np.angle(stft_noisy)
        stft_pred = lin_pred_mag * np.exp(1j * phase_noisy)

        denoised = librosa.istft(
            stft_pred,
            hop_length=dataset.conf.hop_length,
            win_length=dataset.conf.win_length,
            length=len(c_wav)
        )

    return c_wav, n_wav, denoised


def run_training_pipeline(config):
    # 1) Create dataset & dataloader
    dataset_2s = TwoSecDataset(
        clean_dir=CLEAN_2SEC_DIR,
        noisy_dir=NOISY_2SEC_DIR,
        conf=config
    )
    loader = DataLoader(dataset_2s, batch_size=config.batch_size, shuffle=True)

    # 2) Model + optimizer
    model = SimpleUNet().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=config.lr)

    # 3) Train
    for epoch in range(config.epochs):
        avg_loss = train_one_epoch(model, loader, optimizer, config, dataset_2s, epoch_idx=epoch+1)
        print(f"[Epoch {epoch+1}/{config.epochs}] Loss: {avg_loss:.5f}")

        # Quick check
        if (epoch+1)%2==0:
            c, n, d = validate_on_sample(model, dataset_2s, idx=0)
            print("CLEAN:")
            display(Audio(c, rate=config.sr))
            print("NOISY:")
            display(Audio(n, rate=config.sr))
            print("DENOISED:")
            display(Audio(d, rate=config.sr))
            print("--------")

    return model, dataset_2s


# Actually run training
if __name__ == "__main__":
    model_2s, dataset_2s = run_training_pipeline(config)


Using device: cuda
Found 8455 matched 2s pairs.
Computing global min/max across STFT magnitudes ...


100%|██████████| 200/200 [01:16<00:00,  2.62it/s]


  => global min=0.0000, max=4.4869


[Epoch 1/10] Loss: 0.04024


[Epoch 2/10] Loss: 0.03023
CLEAN:


...

[Epoch 9/10] Loss: 0.00783


[Epoch 10/10] Loss: 0.00771
CLEAN:


NOISY:


DENOISED:


--------


In [ ]:
import json

MODEL_NAME = "model_2s_10_epoch"
MODEL_PATH = os.path.join(BASE_PATH, "models", f"{MODEL_NAME}.pth")

torch.save(model_2s.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

JSON_PATH = os.path.join(BASE_PATH, "models", f"{MODEL_NAME}_params.json")

params_dict = {
    "global_min": float(dataset_2s.global_min),
    "global_max": float(dataset_2s.global_max),
    "sr": config.sr,
    "n_fft": config.n_fft,
    "hop_length": config.hop_length,
    "win_length": config.win_length
}
with open(JSON_PATH, "w") as f:
    json.dump(params_dict, f, indent=2)

print(f"Model parameters saved to {JSON_PATH}")

Model saved to /content/drive/MyDrive/colab-data/audio-denoising/models/model_2s_10_epoch.pth
Model parameters saved to /content/drive/MyDrive/colab-data/audio-denoising/models/model_2s_10_epoch_params.json


# Initial Testing

In [4]:
class SimpleUNet(nn.Module):
    """
    Same "time-downsampling" U-Net
    """
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2)
        )
        self.pool1 = nn.MaxPool2d(kernel_size=(1,2))

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1), nn.LeakyReLU(0.2)
        )
        self.pool2 = nn.MaxPool2d(kernel_size=(1,2))

        self.bottleneck = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 128, 3, padding=1), nn.LeakyReLU(0.2),
        )

        self.up2 = nn.Conv2d(128, 64, 1)
        self.dec2 = nn.Sequential(
            nn.Conv2d(64+64, 64, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, 3, padding=1), nn.LeakyReLU(0.2),
        )

        self.up1 = nn.Conv2d(64, 32, 1)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32+32, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 32, 3, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        u2_up = F.interpolate(u2, size=(e2.shape[2], e2.shape[3]), mode='nearest')
        d2 = self.dec2(torch.cat([u2_up, e2], dim=1))

        u1 = self.up1(d2)
        u1_up = F.interpolate(u1, size=(e1.shape[2], e1.shape[3]), mode='nearest')
        d1 = self.dec1(torch.cat([u1_up, e1], dim=1))
        return d1

In [12]:
def load_model_and_params(model_path, josn_path, device="cuda"):
    """
    1) Reads model_name.json for normalization & STFT params
    2) Builds the same U-Net architecture
    3) Loads weights from model_name.pth
    4) Returns (model, params_dict)
    """
    MODEL_PATH = model_path
    JSON_PATH  = josn_path

    # Load JSON
    with open(JSON_PATH, "r") as f:
        params = json.load(f)

    # Build model
    model = SimpleUNet()
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()

    return model, params


def denoise_file_with_model(
    model,
    params,
    noisy_audio_path,
    out_path=None
):
    """
    Given:
      - model (SimpleUNet)
      - params: dict with global_min, global_max, sr, n_fft, hop_length, win_length
      - a single noisy_audio_path
    => returns denoised waveform (numpy)
       optionally saves to out_path
    """
    device = next(model.parameters()).device

    # 1) Load audio
    sr = params["sr"]
    y_noisy, sr_loaded = librosa.load(noisy_audio_path, sr=sr)
    if sr_loaded != sr:
        print(f"Warning: loaded sample rate {sr_loaded}, expected {sr}.")

    # 2) STFT => log1p => scale
    D_noisy = librosa.stft(
        y_noisy,
        n_fft=params["n_fft"],
        hop_length=params["hop_length"],
        win_length=params["win_length"]
    )
    phase_noisy = np.angle(D_noisy)
    mag_noisy = np.abs(D_noisy)
    logmag_noisy = np.log1p(mag_noisy)

    gmin = params["global_min"]
    gmax = params["global_max"]
    logmag_noisy_norm = (logmag_noisy - gmin) / (1e-8 + (gmax - gmin))

    # shape => [1,1,freq,time]
    inp = torch.from_numpy(logmag_noisy_norm).unsqueeze(0).unsqueeze(0).float().to(device)

    # 3) Inference
    with torch.no_grad():
        pred_norm = model(inp)  # => [1,1,freq,time]
        pred_norm = pred_norm.squeeze(0).squeeze(0).cpu().numpy()

    # 4) Unscale => exponent => predicted magnitude
    pred_logmag = pred_norm * (gmax - gmin) + gmin
    pred_mag = np.expm1(pred_logmag)

    # 5) Recombine with noisy phase, do ISTFT
    stft_pred = pred_mag * np.exp(1j * phase_noisy)
    denoised = librosa.istft(
        stft_pred,
        hop_length=params["hop_length"],
        win_length=params["win_length"],
        length=len(y_noisy)
    )

    # 6) Optionally save
    if out_path:
        sf.write(out_path, denoised, sr)
        print(f"Denoised audio saved to {out_path}")

    return denoised, y_noisy

MODEL_NAME = "model_2s_10_epoch"
MODEL_PATH = os.path.join(BASE_PATH, "models", f"{MODEL_NAME}.pth")
JSON_PATH = os.path.join(BASE_PATH, "models", f"{MODEL_NAME}_params.json")
NOISY_2SEC_DIR = os.path.join(BASE_PATH, "data", "two-sec-sample", "noisy")

# 1) Load the model & params from disk
model, params = load_model_and_params(MODEL_PATH, JSON_PATH, device="cuda")

noisy_audio_file_name = os.listdir(NOISY_2SEC_DIR)[100]
noisy_audio_path = os.path.join(NOISY_2SEC_DIR, noisy_audio_file_name)
print(f"Denoising {noisy_audio_path}")


# 2) Denoise a single file
denoised_waveform, noisy_waveform = denoise_file_with_model(
    model,
    params,
    noisy_audio_path=noisy_audio_path,
    out_path="denoised_output.flac"  # optional
)

# 3) Listen in a notebook:
from IPython.display import Audio, display

print("NOISY:")
display(Audio(noisy_waveform, rate=params["sr"]))

print("DENOISED:")
display(Audio(denoised_waveform, rate=params["sr"]))

Denoising /content/drive/MyDrive/colab-data/audio-denoising/data/two-sec-sample/noisy/8230-279154-0034_seg1.flac
Denoised audio saved to denoised_output.flac
NOISY:


DENOISED:
